# Smart Crop Care — DenseNet121 on the common split (Kaggle, full dataset)

This notebook trains **DenseNet121** for the Smart Crop Care project on the **full** Kaggle
*New Plant Diseases Dataset* (`train/` + `valid/`, 87,867 images, 38 classes).

It is one of **three notebooks** (DenseNet121, EfficientNetB0, ResNet50) that are identical except
for the model. All three:
- build **exactly the same train / validation / test split** (files sorted first, seed 42), and print
  a **split fingerprint**. The three fingerprints must be equal, otherwise the comparison is not valid;
- run a **data-leakage check** that must report 0 on every row before training starts;
- use the same image size, augmentation, head, epochs, batch size and optimizer.

**Split method (leaf-grouped, 80 / 10 / 10).** Images of the same physical leaf are put in one group:
same leaf name or UUID in the file name, same MD5 (exact copy), same perceptual hash, **or a
perceptual hash at most 4 bits different** (a near-identical photo, e.g. the same leaf photographed
twice). Each whole group goes to only one split, so no leaf appears in both training and test.

### Before running
1. Click **Add Input**, search for **New Plant Diseases Dataset** (by *vipoooool*), and add it.
2. In **Settings**: **Accelerator → GPU** (T4 ×2 or P100) and **Internet → On**.
3. Use **Save Version → Save & Run All** so training continues if the browser closes.

## Step 1 — Install and imports

In [7]:
!pip install -q imagehash

import os, re, io, json, time, hashlib, shutil, random
from concurrent.futures import ProcessPoolExecutor
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import imagehash
from PIL import Image, ImageOps
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, confusion_matrix)

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("\nSTEP 1 COMPLETED")

TensorFlow: 2.20.0 | Keras: 3.13.2
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

STEP 1 COMPLETED


## Step 2 — Settings

Do not change these: they must be the same in all three notebooks for a fair comparison.
The training settings are the project's own (5 + 5 epochs, batch 32, Adam).

In [8]:
MODEL_NAME = "DenseNet121"               # this notebook's model (the other two notebooks train the other models)

MODELS = {
    # builder, its own preprocessing, first layer to fine-tune in stage 2
    "EfficientNetB0": (keras.applications.EfficientNetB0,
                       keras.applications.efficientnet.preprocess_input, "block6a_expand_conv"),
    "ResNet50":       (keras.applications.ResNet50,
                       keras.applications.resnet50.preprocess_input, "conv5_block1_1_conv"),
    "DenseNet121":    (keras.applications.DenseNet121,
                       keras.applications.densenet.preprocess_input, "conv5_block1_0_bn"),
}
assert MODEL_NAME in MODELS, f"MODEL_NAME must be one of {list(MODELS)}"
BACKBONE, PREPROCESS, FINETUNE_FROM = MODELS[MODEL_NAME]
# Note: EfficientNetB0 contains its own rescaling/normalisation, so its preprocess_input
# is a pass-through: it expects raw pixels in [0, 255].

IMAGE_SIZE = (224, 224)
CHANNELS = 3
BATCH_SIZE = 32
SPLIT_RATIOS = {"train": 0.80, "val": 0.10, "test": 0.10}
NEAR_DUP_BITS = 4                      # pHash difference that still counts as the same leaf photo
AUGMENT = {"rotation": 0.10, "zoom": 0.10, "translation": 0.10, "brightness": 0.10, "contrast": 0.10}
TRAIN = {"dropout": 0.3, "dense_units": 256,
         "stage1_epochs": 5, "stage1_lr": 1e-3,
         "stage2_epochs": 5, "stage2_lr": 1e-5,
         "early_stopping_patience": 5, "reduce_lr_patience": 2,
         "reduce_lr_factor": 0.2, "min_lr": 1e-7}

TAG = MODEL_NAME.lower()
OUT = f"/kaggle/working/smart_crop_care_{TAG}"
MODEL_DIR = f"{OUT}/models/{TAG}"
REPORTS = f"{OUT}/artifacts/reports"
PLOTS = f"{OUT}/artifacts/plots"
SPLITS_OUT = f"{OUT}/data/splits"
for d in (MODEL_DIR, REPORTS, PLOTS, SPLITS_OUT):
    os.makedirs(d, exist_ok=True)
BEST_MODEL = f"{MODEL_DIR}/best_{TAG}.keras"
print("Model:", MODEL_NAME, "| fine-tune from:", FINETUNE_FROM, "| output:", OUT)
print("\nSTEP 2 COMPLETED")

Model: DenseNet121 | fine-tune from: conv5_block1_0_bn | output: /kaggle/working/smart_crop_care_densenet121

STEP 2 COMPLETED


## Step 3 — Find the dataset and list all images

In [9]:
def find_split_dir(name):
    """The folder called `name` (train/valid) that contains the class sub-folders."""
    for root, dirs, _ in os.walk("/kaggle/input"):
        if os.path.basename(root) == name and len(dirs) >= 30:
            return root
    raise FileNotFoundError(f"No '{name}' folder with class sub-folders under /kaggle/input. Add the dataset.")

TRAIN_DIR, VALID_DIR = find_split_dir("train"), find_split_dir("valid")
print("train:", TRAIN_DIR)
print("valid:", VALID_DIR)

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
files = []
for source, base in (("train", TRAIN_DIR), ("valid", VALID_DIR)):
    for label in sorted(os.listdir(base)):
        folder = os.path.join(base, label)
        if os.path.isdir(folder):
            for f in os.listdir(folder):
                if f.lower().endswith(IMG_EXT):
                    files.append((os.path.join(folder, f), source, label, f))
# Sorting makes the split independent of the order the file system lists files in,
# so all three notebooks build exactly the same split.
inv = (pd.DataFrame(files, columns=["path", "source_split", "label", "filename"])
         .sort_values(["label", "source_split", "filename"]).reset_index(drop=True))
print(inv.groupby("source_split").size().to_string())
print("Total images:", len(inv), "| classes:", inv["label"].nunique())
assert inv["label"].nunique() == 38
# The 33 images in the supplied test/ folder are copies of valid/ images, so they are not used.
print("\nSTEP 3 COMPLETED")

train: /kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train
valid: /kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid
source_split
train    70295
valid    17572
Total images: 87867 | classes: 38

STEP 3 COMPLETED


## Step 4 — Hash every image (to find duplicates and copies of the same leaf)
This takes several minutes.

In [10]:
def inspect(path):
    rec = {"path": path, "md5": None, "phash": None, "format": None, "error": None}
    try:
        with open(path, "rb") as fh:
            data = fh.read()
        rec["md5"] = hashlib.md5(data).hexdigest()
        with Image.open(io.BytesIO(data)) as im:
            im.verify()
        with Image.open(io.BytesIO(data)) as im:
            im.load()
            rec["format"] = im.format
            rec["phash"] = str(imagehash.phash(im))
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {e}"
    return rec

t0 = time.time()
with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    recs = list(ex.map(inspect, inv["path"], chunksize=256))
inv = inv.merge(pd.DataFrame(recs), on="path")
print(f"Hashed {len(inv):,} images in {time.time() - t0:.0f} s")
bad = inv[inv["error"].notna() | (inv["format"] != "JPEG")]
print("Unreadable or non-JPEG images removed:", len(bad))
inv = inv.drop(bad.index).reset_index(drop=True)
print("\nSTEP 4 COMPLETED")

Hashed 87,867 images in 64 s
Unreadable or non-JPEG images removed: 0

STEP 4 COMPLETED


## Step 5 — Leaf-grouped, leakage-free split (80 / 10 / 10)

Images are grouped when they share the leaf name or UUID in the file name, the same MD5,
the same perceptual hash, or a perceptual hash at most `NEAR_DUP_BITS` (4) bits different within
the same class. Each whole group goes to only one split, so no leaf appears in both training and test.

In [11]:
AUG_SUFFIX = re.compile(r"_(flipLR|flipTB|\d+deg|new\w*)$", re.IGNORECASE)
UUID = re.compile(r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$", re.I)

def parse_name(filename):
    stem = filename.rsplit(".", 1)[0]
    uuid = None
    if "___" in stem:
        prefix, rest = stem.split("___", 1)
        if UUID.match(prefix):
            uuid, stem = prefix.lower(), rest
    return uuid, AUG_SUFFIX.sub("", stem).strip().lower()

parsed = inv["filename"].map(parse_name)
inv["uuid"], inv["leaf_key"] = parsed.str[0], parsed.str[1]

before = len(inv)
inv = inv.drop_duplicates("md5", keep="first").reset_index(drop=True)
print("Exact duplicates removed:", before - len(inv))

BITS = np.array([bin(i).count("1") for i in range(256)], np.uint8)

def hamming(a, b):
    """Bit differences between every hash in a (n,) and b (m,), both uint64 -> (n, m)."""
    x = (a[:, None] ^ b[None, :]).view(np.uint8).reshape(len(a), len(b), 8)
    return BITS[x].sum(-1)

def near_duplicate_pairs(df, max_bits=NEAR_DUP_BITS, chunk=512):
    """Index pairs (i, j), i < j, in the same class whose pHash differs by at most max_bits."""
    hashes = df["phash"].map(lambda s: int(s, 16)).to_numpy(np.uint64)
    pairs = []
    for _, idx in df.groupby("label").indices.items():
        h = hashes[idx]
        for s in range(0, len(idx), chunk):
            a, b = np.nonzero(hamming(h[s:s + chunk], h) <= max_bits)
            keep = (a + s) < b
            pairs += list(zip(idx[a[keep] + s], idx[b[keep]]))
    return pairs

def assign_groups(df):
    parent = np.arange(len(df))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[max(ri, rj)] = min(ri, rj)
    for column in ["leaf_key", "uuid", "md5", "phash"]:
        for _, idx in df.groupby(["label", column], dropna=True).indices.items():
            for j in idx[1:]:
                union(idx[0], j)
    near = near_duplicate_pairs(df)
    for i, j in near:
        union(i, j)
    print(f"Near-identical photo pairs (pHash <= {NEAR_DUP_BITS} bits, same class): {len(near):,}")
    roots = np.array([find(i) for i in range(len(df))])
    return np.unique(roots, return_inverse=True)[1]

def split_groups(df, ratios=SPLIT_RATIOS, seed=SEED):
    rng = np.random.default_rng(seed)
    names, assignment = list(ratios), {}
    for label in sorted(df["label"].unique()):
        sizes = df[df["label"] == label].groupby("group_id").size()
        target = {s: ratios[s] * sizes.sum() for s in names}
        filled = dict.fromkeys(names, 0)
        for gid in rng.permutation(sizes.index.to_numpy()):
            s = max(names, key=lambda n: (target[n] - filled[n]) / target[n])
            assignment[gid] = s
            filled[s] += sizes[gid]
    return df["group_id"].map(assignment)

inv["group_id"] = assign_groups(inv)
inv["split"] = split_groups(inv)
CLASS_NAMES = sorted(inv["label"].unique())
inv["label_idx"] = inv["label"].map({c: i for i, c in enumerate(CLASS_NAMES)})

def fingerprint(split):
    """Short hash of the sorted MD5s in a split: equal fingerprints = identical images."""
    return hashlib.sha256("\n".join(sorted(inv.loc[inv["split"] == split, "md5"])).encode()).hexdigest()[:16]

FINGERPRINT = {s: fingerprint(s) for s in SPLIT_RATIOS}
share = inv.groupby("label")["split"].value_counts(normalize=True).unstack().fillna(0)
split_report = {
    "seed": SEED, "ratios": SPLIT_RATIOS, "near_duplicate_bits": NEAR_DUP_BITS,
    "images_in_pool": int(before),
    "exact_duplicates_dropped": int(before - len(inv)), "images_after_dedup": int(len(inv)),
    "num_groups": int(inv["group_id"].nunique()),
    "largest_group": int(inv.groupby("group_id").size().max()),
    "split_sizes": inv["split"].value_counts().reindex(list(SPLIT_RATIOS)).astype(int).to_dict(),
    "test_share_per_class": {"min": round(float(share["test"].min()), 4), "max": round(float(share["test"].max()), 4)},
    "fingerprint": FINGERPRINT,
    "dataset": "New Plant Diseases Dataset (full Kaggle version, train/ + valid/)",
}
json.dump(split_report, open(f"{SPLITS_OUT}/split_report.json", "w"), indent=2)
for s in SPLIT_RATIOS:
    inv[inv["split"] == s][["path", "label", "label_idx", "group_id", "split", "md5", "phash"]].to_csv(
        f"{SPLITS_OUT}/{s}.csv", index=False)
print(json.dumps({k: split_report[k] for k in ("split_sizes", "num_groups", "largest_group", "test_share_per_class")}, indent=2))
print("\nSPLIT FINGERPRINT (must be the same in all three notebooks):")
for s, f in FINGERPRINT.items():
    print(f"  {s:5s} {f}")
print("\nSTEP 5 COMPLETED")

Exact duplicates removed: 26
Near-identical photo pairs (pHash <= 4 bits, same class): 1,934
{
  "split_sizes": {
    "train": 70138,
    "val": 8904,
    "test": 8799
  },
  "num_groups": 45181,
  "largest_group": 354,
  "test_share_per_class": {
    "min": 0.0987,
    "max": 0.1018
  }
}

SPLIT FINGERPRINT (must be the same in all three notebooks):
  train d4d22ddbaf9cd66f
  val   9df46ddb974db199
  test  13afce0ee189dd7d

STEP 5 COMPLETED


## Step 5b — Data-leakage check

Every row must be **0**. If any row is not 0, the notebook stops here, before training.

In [12]:
leak = {}

def across_splits(column, per_class=True):
    """How many values of `column` appear in more than one split."""
    keys = ["label", column] if per_class else [column]
    d = inv.dropna(subset=[column])
    return int((d.groupby(keys)["split"].nunique() > 1).sum())

leak["Same image file (MD5) in two splits"] = across_splits("md5", per_class=False)
leak["Same leaf group in two splits"] = across_splits("group_id", per_class=False)
leak["Same leaf name in two splits (same class)"] = across_splits("leaf_key")
leak["Same image UUID in two splits (same class)"] = across_splits("uuid")
leak["Same perceptual hash in two splits (any class)"] = across_splits("phash", per_class=False)

near = 0
hashes = inv["phash"].map(lambda s: int(s, 16)).to_numpy(np.uint64)
for _, idx in inv.groupby("label").indices.items():
    sp = inv["split"].to_numpy()[idx]
    train_h = hashes[idx][sp == "train"]
    for other in ("val", "test"):
        h = hashes[idx][sp == other]
        if len(h) and len(train_h):
            near += int((hamming(h, train_h).min(1) <= NEAR_DUP_BITS).sum())
leak[f"Val/test image with a near-identical train image (pHash <= {NEAR_DUP_BITS} bits)"] = near
vt = inv[inv["split"] != "train"]
near_vt = int(sum((hamming(g.loc[g["split"] == "val", "phash"].map(lambda s: int(s, 16)).to_numpy(np.uint64),
                           g.loc[g["split"] == "test", "phash"].map(lambda s: int(s, 16)).to_numpy(np.uint64))
                   <= NEAR_DUP_BITS).any(1).sum()
                  for _, g in vt.groupby("label") if (g["split"] == "val").any() and (g["split"] == "test").any()))
leak[f"Val image with a near-identical test image (pHash <= {NEAR_DUP_BITS} bits)"] = near_vt

# The Kaggle dataset also has a small test/ folder: check that it is only copies of valid/ images.
test_dir = None
for root, dirs, fs in os.walk("/kaggle/input"):
    if os.path.basename(root) == "test" and any(f.lower().endswith(IMG_EXT) for f in fs):
        test_dir = root
        break
supplied = []
if test_dir:
    for f in sorted(os.listdir(test_dir)):
        if f.lower().endswith(IMG_EXT):
            supplied.append(hashlib.md5(open(os.path.join(test_dir, f), "rb").read()).hexdigest())
pool_md5 = set(inv["md5"])

print(f"{'Check':78s} Result")
for name, n in leak.items():
    print(f"{name:78s} {n}")
print(f"\nSupplied test/ folder: {len(supplied)} images, {sum(m in pool_md5 for m in supplied)} are exact "
      "copies of images already in the pool, so the folder is not used.")
json.dump({"checks": leak, "near_duplicate_bits": NEAR_DUP_BITS, "fingerprint": FINGERPRINT,
           "supplied_test_folder": {"images": len(supplied), "copies_in_pool": int(sum(m in pool_md5 for m in supplied))}},
          open(f"{REPORTS}/leakage_check.json", "w"), indent=2)
assert all(n == 0 for n in leak.values()), "DATA LEAKAGE FOUND - do not train; check the split."
print("\nNO DATA LEAKAGE FOUND")
print("\nSTEP 5b COMPLETED")

Check                                                                          Result
Same image file (MD5) in two splits                                            0
Same leaf group in two splits                                                  0
Same leaf name in two splits (same class)                                      0
Same image UUID in two splits (same class)                                     0
Same perceptual hash in two splits (any class)                                 0
Val/test image with a near-identical train image (pHash <= 4 bits)             0
Val image with a near-identical test image (pHash <= 4 bits)                   0

Supplied test/ folder: 33 images, 33 are exact copies of images already in the pool, so the folder is not used.

NO DATA LEAKAGE FOUND

STEP 5b COMPLETED


## Step 6 — Input pipeline (same resize and augmentation as the project)

In [13]:
AUTOTUNE = tf.data.AUTOTUNE

def resize_image(image):
    image = tf.image.resize(image, IMAGE_SIZE, method="bilinear", antialias=True)
    image = tf.clip_by_value(image, 0.0, 255.0)
    image.set_shape((*IMAGE_SIZE, CHANNELS))
    return image

def load_image(path):
    data = tf.io.read_file(path)
    image = tf.io.decode_jpeg(data, channels=CHANNELS, dct_method="INTEGER_ACCURATE")
    return resize_image(image)

def build_augmenter(seed=SEED):
    a = AUGMENT
    return keras.Sequential([
        keras.layers.RandomFlip("horizontal_and_vertical", seed=seed),
        keras.layers.RandomRotation(a["rotation"], fill_mode="reflect", seed=seed),
        keras.layers.RandomZoom(a["zoom"], fill_mode="reflect", seed=seed),
        keras.layers.RandomTranslation(a["translation"], a["translation"], fill_mode="reflect", seed=seed),
        keras.layers.RandomBrightness(a["brightness"], value_range=(0, 255), seed=seed),
        keras.layers.RandomContrast(a["contrast"], seed=seed),
    ], name="augmentation")

def make_dataset(split, training=None, batch_size=BATCH_SIZE):
    training = (split == "train") if training is None else training
    df = inv[inv["split"] == split]
    ds = tf.data.Dataset.from_tensor_slices((df["path"].tolist(), df["label_idx"].to_numpy("int32")))
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: (load_image(p), y), num_parallel_calls=AUTOTUNE).batch(batch_size)
    if training:
        aug = build_augmenter()
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=AUTOTUNE)
        ds = ds.map(lambda x, y: (tf.clip_by_value(x, 0.0, 255.0), y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (PREPROCESS(x), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds, val_ds = make_dataset("train"), make_dataset("val")
x, y = next(iter(train_ds))
print("Batch:", x.shape, x.dtype, "| value range:", float(tf.reduce_min(x)), "to", float(tf.reduce_max(x)))
print("\nSTEP 6 COMPLETED")

I0000 00:00:1793284709.533313      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793284709.541696      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Batch: (32, 224, 224, 3) <dtype: 'float32'> | value range: -2.1179039478302 to 2.5842301845550537

STEP 6 COMPLETED


## Step 7 — Build the model (same head and layer names as the project)

In [14]:
def build_model(num_classes):
    inputs = keras.Input(shape=(*IMAGE_SIZE, CHANNELS), name="image")
    backbone = BACKBONE(include_top=False, weights="imagenet", input_shape=inputs.shape[1:])
    backbone.trainable = False
    x = backbone(inputs, training=False)              # BatchNorm stays in inference mode
    x = keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = keras.layers.Dropout(TRAIN["dropout"], name="dropout")(x)
    x = keras.layers.Dense(TRAIN["dense_units"], activation="relu", name="dense")(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    return keras.Model(inputs, outputs, name=f"{TAG}_plant_disease"), backbone

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy",
                  metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3")])

def callbacks(stage, best_so_far=None):
    return [
        keras.callbacks.ModelCheckpoint(BEST_MODEL, monitor="val_accuracy", mode="max", save_best_only=True,
                                        initial_value_threshold=best_so_far, verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=TRAIN["early_stopping_patience"],
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=TRAIN["reduce_lr_factor"],
                                          patience=TRAIN["reduce_lr_patience"], min_lr=TRAIN["min_lr"], verbose=1),
        keras.callbacks.CSVLogger(f"{REPORTS}/training_{stage}.csv"),
    ]

model, backbone = build_model(len(CLASS_NAMES))
np.save(f"{MODEL_DIR}/class_names.npy", np.array(CLASS_NAMES))
json.dump(CLASS_NAMES, open(f"{MODEL_DIR}/class_names.json", "w"), indent=2)
print("Model:", model.name, "| total parameters:", f"{model.count_params():,}")
assert model.output_shape == (None, 38)
print("\nSTEP 7 COMPLETED")

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Model: densenet121_plant_disease | total parameters: 7,309,670

STEP 7 COMPLETED


## Step 8 — Stage 1: train the new head (backbone frozen)

In [15]:
compile_model(model, TRAIN["stage1_lr"])
t0 = time.time()
h1 = model.fit(train_ds, validation_data=val_ds, epochs=TRAIN["stage1_epochs"],
               callbacks=callbacks("stage1"), verbose=1)
STAGE_MIN = {"stage1": round((time.time() - t0) / 60, 1)}
print(f"\nStage 1 finished in {STAGE_MIN['stage1']} min | best val accuracy:",
      round(max(h1.history["val_accuracy"]), 4))

Epoch 1/5
   1/2192 ━━━━━━━━━━━━━━━━━━━━ 17:37:22 29s/step - accuracy: 0.0312 - loss: 3.9250 - top3: 0.0938

I0000 00:00:1793284748.928480     183 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2192/2192 ━━━━━━━━━━━━━━━━━━━━ 0s 272ms/step - accuracy: 0.6783 - loss: 1.1191 - top3: 0.8425
Epoch 1: val_accuracy improved from None to 0.91622, saving model to /kaggle/working/smart_crop_care_densenet121/models/densenet121/best_densenet121.keras

Epoch 1: finished saving model to /kaggle/working/smart_crop_care_densenet121/models/densenet121/best_densenet121.keras
2192/2192 ━━━━━━━━━━━━━━━━━━━━ 669s 292ms/step - accuracy: 0.7854 - loss: 0.6970 - top3: 0.9276 - val_accuracy: 0.9162 - val_loss: 0.2594 - val_top3: 0.9867 - learning_rate: 0.0010
Epoch 2/5
2192/2192 ━━━━━━━━━━━━━━━━━━━━ 0s 265ms/step - accuracy: 0.8571 - loss: 0.4351 - top3: 0.9707
Epoch 2: val_accuracy did not improve from 0.91622
2192/2192 ━━━━━━━━━━━━━━━━━━━━ 603s 275ms/step - accuracy: 0.8614 - loss: 0.4213 - top3: 0.9720 - val_accuracy: 0.9104 - val_loss: 0.2589 - val_top3: 0.9853 - learning_rate: 0.0010
Epoch 3/5
2192/2192 ━━━━━━━━━━━━━━━━━━━━ 0s 259ms/step - accuracy: 0.8728 - loss: 0.3816 - top3: 0.9765
Epoch 3: 

## Step 9 — Stage 2: fine-tune the last blocks with a small learning rate

In [16]:
names = [l.name for l in backbone.layers]
assert FINETUNE_FROM in names, f"{FINETUNE_FROM} not found in {MODEL_NAME}"
start = names.index(FINETUNE_FROM)
backbone.trainable = True
for i, layer in enumerate(backbone.layers):
    layer.trainable = i >= start and not isinstance(layer, keras.layers.BatchNormalization)

compile_model(model, TRAIN["stage2_lr"])
print("Trainable parameters in stage 2:", f"{sum(int(np.prod(w.shape)) for w in model.trainable_weights):,}")
t0 = time.time()
h2 = model.fit(train_ds, validation_data=val_ds, epochs=TRAIN["stage2_epochs"],
               callbacks=callbacks("stage2", best_so_far=max(h1.history["val_accuracy"])), verbose=1)
STAGE_MIN["stage2"] = round((time.time() - t0) / 60, 1)
print(f"\nStage 2 finished in {STAGE_MIN['stage2']} min | best val accuracy:",
      round(max(h2.history["val_accuracy"]), 4))
json.dump({"stage1": h1.history, "stage2": h2.history, "config": TRAIN, "batch_size": BATCH_SIZE,
           "model": MODEL_NAME, "train_minutes": STAGE_MIN, "gpus": len(gpus)},
          open(f"{REPORTS}/training_history.json", "w"), indent=2, default=float)

Trainable parameters in stage 2: 2,402,086
Epoch 1/5
2192/2192 ━━━━━━━━━━━━━━━━━━━━ 0s 274ms/step - accuracy: 0.9187 - loss: 0.2382 - top3: 0.9888
Epoch 1: val_accuracy improved from 0.93160 to 0.95620, saving model to /kaggle/working/smart_crop_care_densenet121/models/densenet121/best_densenet121.keras

Epoch 1: finished saving model to /kaggle/working/smart_crop_care_densenet121/models/densenet121/best_densenet121.keras
2192/2192 ━━━━━━━━━━━━━━━━━━━━ 674s 293ms/step - accuracy: 0.9288 - loss: 0.2091 - top3: 0.9913 - val_accuracy: 0.9562 - val_loss: 0.1314 - val_top3: 0.9956 - learning_rate: 1.0000e-05
Epoch 2/5
2192/2192 ━━━━━━━━━━━━━━━━━━━━ 0s 260ms/step - accuracy: 0.9437 - loss: 0.1630 - top3: 0.9945
Epoch 2: val_accuracy improved from 0.95620 to 0.96047, saving model to /kaggle/working/smart_crop_care_densenet121/models/densenet121/best_densenet121.keras

Epoch 2: finished saving model to /kaggle/working/smart_crop_care_densenet121/models/densenet121/best_densenet121.keras
2192/2

## Step 10 — Final evaluation (best saved model)

This reports accuracy, macro precision, macro recall, macro F1, weighted F1, macro ROC-AUC and
top-3 accuracy for each split. **The test numbers are the ones to report.**

In [17]:
best = keras.models.load_model(BEST_MODEL, compile=False)

def evaluate(split):
    df = inv[inv["split"] == split]
    y = df["label_idx"].to_numpy()
    probs = best.predict(make_dataset(split, training=False), verbose=1)
    pred = probs.argmax(1)
    top3 = float(np.mean([y[i] in np.argsort(probs[i])[-3:] for i in range(len(y))]))
    m = {"images": int(len(y)),
         "accuracy": accuracy_score(y, pred),
         "precision_macro": precision_score(y, pred, average="macro", zero_division=0),
         "recall_macro": recall_score(y, pred, average="macro", zero_division=0),
         "f1_macro": f1_score(y, pred, average="macro", zero_division=0),
         "precision_weighted": precision_score(y, pred, average="weighted", zero_division=0),
         "recall_weighted": recall_score(y, pred, average="weighted", zero_division=0),
         "f1_weighted": f1_score(y, pred, average="weighted", zero_division=0),
         "roc_auc_macro": roc_auc_score(y, probs, multi_class="ovr", average="macro"),
         "top3_accuracy": top3,
         "loss": float(-np.mean(np.log(np.clip(probs[np.arange(len(y)), y], 1e-12, 1))))}
    return y, probs, pred, {k: (float(v) if k != "images" else v) for k, v in m.items()}

results = {}
for split in ("train", "val", "test"):
    y, probs, pred, results[split] = evaluate(split)
    print(split, {k: round(v, 4) for k, v in results[split].items()})
    if split == "test":
        y_test, probs_test, pred_test = y, probs, pred

np.save(f"{REPORTS}/test_probabilities.npy", probs_test)
rep = classification_report(y_test, pred_test, labels=range(38), target_names=CLASS_NAMES, output_dict=True, zero_division=0)
open(f"{REPORTS}/classification_report.txt", "w").write(
    classification_report(y_test, pred_test, labels=range(38), target_names=CLASS_NAMES, digits=4, zero_division=0))
cm = confusion_matrix(y_test, pred_test, labels=range(38))
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(f"{REPORTS}/confusion_matrix_test.csv")

per_class = {c: {k: rep[c][k] for k in ("precision", "recall", "f1-score", "support")} for c in CLASS_NAMES}
worst = sorted(((c, v["f1-score"]) for c, v in per_class.items()), key=lambda t: t[1])[:5]
metrics = {"generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           "model": MODEL_NAME, "model_path": f"models/{TAG}/best_{TAG}.keras", "num_classes": 38,
           "evaluation": results,
           "training_epochs": {"stage1": len(h1.history["loss"]), "stage2": len(h2.history["loss"])},
           "test_per_class": per_class, "test_lowest_f1_classes": dict(worst),
           "split_fingerprint": FINGERPRINT, "train_minutes": STAGE_MIN,
           "notes": {"dataset": "Full Kaggle New Plant Diseases Dataset, leaf-grouped 80/10/10 split "
                                "with near-duplicate grouping (seed 42)",
                     "evaluation": "Clean evaluation of the best checkpoint (no augmentation, no dropout)."}}
json.dump(metrics, open(f"{REPORTS}/model_metrics.json", "w"), indent=2)
print("\nLowest test F1:", [(c, round(f, 3)) for c, f in worst])
print("\nSTEP 10 COMPLETED")

2192/2192 ━━━━━━━━━━━━━━━━━━━━ 191s 82ms/step
train {'images': 70138, 'accuracy': 0.9767, 'precision_macro': 0.9774, 'recall_macro': 0.9765, 'f1_macro': 0.9766, 'precision_weighted': 0.9776, 'recall_weighted': 0.9767, 'f1_weighted': 0.9768, 'roc_auc_macro': 0.9999, 'top3_accuracy': 0.9986, 'loss': 0.0668}
279/279 ━━━━━━━━━━━━━━━━━━━━ 30s 106ms/step
val {'images': 8904, 'accuracy': 0.9686, 'precision_macro': 0.97, 'recall_macro': 0.9681, 'f1_macro': 0.9685, 'precision_weighted': 0.9694, 'recall_weighted': 0.9686, 'f1_weighted': 0.9684, 'roc_auc_macro': 0.9998, 'top3_accuracy': 0.9983, 'loss': 0.0914}
275/275 ━━━━━━━━━━━━━━━━━━━━ 34s 122ms/step
test {'images': 8799, 'accuracy': 0.9695, 'precision_macro': 0.9705, 'recall_macro': 0.9695, 'f1_macro': 0.9695, 'precision_weighted': 0.9706, 'recall_weighted': 0.9695, 'f1_weighted': 0.9696, 'roc_auc_macro': 0.9997, 'top3_accuracy': 0.9981, 'loss': 0.0885}

Lowest test F1: [('Tomato___Target_Spot', 0.858), ('Tomato___Early_blight', 0.874), ('Tom

## Step 11 — Results table (same columns as the comparison slide)
Each of the three notebooks saves its own row to `comparison_row.csv`; combine them after all three runs.

In [18]:
t = results["test"]
row = pd.DataFrame([{
    "Model": MODEL_NAME,
    "Dataset": "Leaf-grouped test split (full Kaggle data)",
    "Images": t["images"],
    "Accuracy (%)": round(100 * t["accuracy"], 2),
    "Macro Precision (%)": round(100 * t["precision_macro"], 2),
    "Macro Recall (%)": round(100 * t["recall_macro"], 2),
    "Macro F1 (%)": round(100 * t["f1_macro"], 2),
    "Weighted F1 (%)": round(100 * t["f1_weighted"], 2),
    "Macro ROC-AUC (%)": round(100 * t["roc_auc_macro"], 2),
    "Top-3 (%)": round(100 * t["top3_accuracy"], 2),
    "Parameters": f"{best.count_params():,}",
    "Training time (min)": round(sum(STAGE_MIN.values()), 1),
    "Split fingerprint (test)": FINGERPRINT["test"],
}])
row.to_csv(f"{REPORTS}/comparison_row.csv", index=False)
row

,Model,Dataset,Images,Accuracy (%),Macro Precision (%),Macro Recall (%),Macro F1 (%),Weighted F1 (%),Macro ROC-AUC (%),Top-3 (%),Parameters,Training time (min),Split fingerprint (test)
0,DenseNet121,Leaf-grouped test split (full Kaggle data),8799,96.95,97.05,96.95,96.95,96.96,99.97,99.81,"7,309,670",102.9,13afce0ee189dd7d


## Step 12 — Accuracy and loss curves, confusion matrix

In [19]:
joined = lambda k: h1.history[k] + h2.history[k]
ep = np.arange(1, len(joined("loss")) + 1)
for key, fname in (("accuracy", "accuracy_curve.png"), ("loss", "loss_curve.png")):
    plt.figure(figsize=(10, 5.2))
    plt.plot(ep, joined(key), "o-", label="train (augmented)")
    plt.plot(ep, joined("val_" + key), "o-", label="validation")
    plt.axvline(len(h1.history["loss"]) + 0.5, ls="--", c="grey")
    plt.title(f"{MODEL_NAME} {key} per epoch", fontweight="bold")
    plt.xlabel("Epoch"); plt.ylabel(key.capitalize()); plt.xticks(ep); plt.grid(alpha=0.3); plt.legend()
    plt.savefig(f"{PLOTS}/{fname}", dpi=130, bbox_inches="tight"); plt.close()

plt.figure(figsize=(14, 12.5))
plt.imshow(cm / cm.sum(1, keepdims=True), cmap="Blues"); plt.colorbar(label="Recall")
plt.xticks(range(38), CLASS_NAMES, rotation=90, fontsize=6); plt.yticks(range(38), CLASS_NAMES, fontsize=6)
plt.title(f"{MODEL_NAME} – test confusion matrix ({len(y_test):,} images, accuracy {results['test']['accuracy']:.4f})")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.savefig(f"{PLOTS}/confusion_matrix.png", dpi=130, bbox_inches="tight"); plt.close()
print("Saved:", sorted(os.listdir(PLOTS)))
print("\nSTEP 12 COMPLETED")

Saved: ['accuracy_curve.png', 'confusion_matrix.png', 'loss_curve.png']

STEP 12 COMPLETED


## Step 13 — Non-leaf reference (needed if this model is ever used in the app)

The app rejects photos that are not leaves by comparing the image's `gap` features with real
training leaves. Every model has different features, so each one needs its own reference file.

In [20]:
K, PER_CLASS, THRESHOLD = 5, 300, 0.60   # 300 reference leaves per class, as the app uses for EfficientNetB0
feature_model = keras.Model(best.input, [best.get_layer("gap").output, best.output])

def app_preprocess(img):
    """Same steps as the app: EXIF fix, RGB, centre square crop, resize 224, model preprocessing."""
    im = ImageOps.exif_transpose(img if isinstance(img, Image.Image) else Image.open(img)).convert("RGB")
    w, h = im.size; s = min(w, h)
    im = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))
    return np.asarray(PREPROCESS(resize_image(np.asarray(im, dtype=np.uint8))))

def features(items, bs=64):
    F, P = [], []
    for i in range(0, len(items), bs):
        f, p = feature_model(np.stack([app_preprocess(x) for x in items[i:i + bs]]), training=False)
        F.append(np.asarray(f)); P.append(np.asarray(p))
    return np.concatenate(F), np.concatenate(P)

def knn_score(feats, ref):
    sims = (feats / np.linalg.norm(feats, axis=1, keepdims=True)) @ ref.T
    return np.sort(sims, axis=1)[:, -K:].mean(1)

sample = lambda split, n: inv[inv["split"] == split].groupby("label").sample(n, random_state=7)
ref_f, _ = features(sample("train", PER_CLASS)["path"].tolist())
ref = ref_f / np.linalg.norm(ref_f, axis=1, keepdims=True)

scope = {"model": MODEL_NAME, "reference_images": int(len(ref)), "k": K}
for split, n in (("val", 50), ("test", 10)):
    f, _ = features(sample(split, n)["path"].tolist())
    scope[split] = {"images": 38 * n, "min_score": round(float(knn_score(f, ref).min()), 4)}

from matplotlib import cbook
from sklearn.datasets import load_sample_images
rng = np.random.default_rng(0)
pics = {"face_photo": Image.open(cbook.get_sample_data("grace_hopper.jpg")),
        "city_scene": Image.fromarray(load_sample_images().images[0]),
        "flower_photo": Image.fromarray(load_sample_images().images[1]),
        "plot": Image.open(f"{PLOTS}/confusion_matrix.png")}
ood = {}
for name, im in pics.items():
    im = im.convert("RGB"); ood[name] = im; w, h = im.size
    for j in range(4):
        side = int(min(w, h) * rng.uniform(0.4, 0.8))
        x0, y0 = rng.integers(0, w - side + 1), rng.integers(0, h - side + 1)
        ood[f"{name}_crop{j}"] = im.crop((x0, y0, x0 + side, y0 + side))
for j in range(5):
    ood[f"noise{j}"] = Image.fromarray(rng.integers(0, 256, (224, 224, 3), dtype=np.uint8))
yy, xx = np.indices((224, 224))
ood["checkerboard"] = Image.fromarray(np.repeat(((yy + xx) // 28 % 2 * 255)[..., None], 3, -1).astype(np.uint8))
ood["stripes"] = Image.fromarray(np.repeat((xx // 16 % 2 * 255)[..., None], 3, -1).astype(np.uint8))
f, p = features(list(ood.values()))
s_ood = knn_score(f, ref)

min_real = min(scope["val"]["min_score"], scope["test"]["min_score"])
max_ood = float(s_ood.max())
if not (max_ood < THRESHOLD <= min_real):
    print(f"Note: 0.60 does not separate real leaves (min {min_real:.3f}) from non-leaf images (max {max_ood:.3f}).")
    if max_ood < min_real:
        THRESHOLD = round((max_ood + min_real) / 2, 3)
        print("Using the midpoint threshold instead:", THRESHOLD)
    else:
        print("WARNING: real leaves and non-leaf images overlap for this model - the check would not be reliable.")
scope.update({"threshold": THRESHOLD, "out_of_scope": {
    "images": len(ood), "max_score": round(max_ood, 4), "caught": int((s_ood < THRESHOLD).sum()),
    "mean_softmax_confidence_without_check": round(float(p.max(1).mean()), 4)}})
np.savez_compressed(f"{MODEL_DIR}/leaf_reference.npz", features=ref.astype(np.float16),
                    threshold=np.float32(THRESHOLD), k=np.int32(K), layer=np.array("gap"),
                    per_class=np.int32(PER_CLASS))
json.dump(scope, open(f"{REPORTS}/leaf_scope_check.json", "w"), indent=2)
print(json.dumps(scope, indent=2))
print("\nSTEP 13 COMPLETED")

{
  "model": "DenseNet121",
  "reference_images": 11400,
  "k": 5,
  "val": {
    "images": 1900,
    "min_score": 0.7137
  },
  "test": {
    "images": 380,
    "min_score": 0.7176
  },
  "threshold": 0.6,
  "out_of_scope": {
    "images": 27,
    "max_score": 0.573,
    "caught": 27,
    "mean_softmax_confidence_without_check": 0.6964
  }
}

STEP 13 COMPLETED


## Step 14 — Sanity check (loads the files the way the app does)

In [21]:
m = keras.models.load_model(BEST_MODEL, compile=False)
names = [str(n) for n in np.load(f"{MODEL_DIR}/class_names.npy", allow_pickle=False)]
assert tuple(m.input_shape) == (None, 224, 224, 3), m.input_shape
assert m.output_shape[-1] == len(names) == 38
assert m.get_layer("gap") is not None
with np.load(f"{MODEL_DIR}/leaf_reference.npz", allow_pickle=False) as d:
    print("leaf_reference:", {k: d[k].shape for k in d.files}, "| threshold", float(d["threshold"]))
row = inv[inv["split"] == "test"].sample(1, random_state=1).iloc[0]
pr = m(app_preprocess(row["path"])[None], training=False).numpy()[0]
print("True:", row["label"], "| Predicted:", names[pr.argmax()], f"({pr.max():.1%})")
print("\nALL CHECKS PASSED")

leaf_reference: {'features': (11400, 1024), 'threshold': (), 'k': (), 'layer': (), 'per_class': ()} | threshold 0.6000000238418579
True: Tomato___Tomato_mosaic_virus | Predicted: Tomato___Tomato_mosaic_virus (100.0%)

ALL CHECKS PASSED


## Step 15 — Zip everything for download
Download the zip from the **Output** panel (or the version's **Output** tab).

In [22]:
zip_path = shutil.make_archive(f"/kaggle/working/smart_crop_care_{TAG}_results", "zip", OUT)
print(zip_path, f"{os.path.getsize(zip_path) / 1e6:.1f} MB")
print("Contents:")
for root, _, fs in os.walk(OUT):
    for f in fs:
        print(" ", os.path.relpath(os.path.join(root, f), OUT))

/kaggle/working/smart_crop_care_densenet121_results.zip 70.6 MB
Contents:
  data/splits/train.csv
  data/splits/split_report.json
  data/splits/test.csv
  data/splits/val.csv
  artifacts/plots/loss_curve.png
  artifacts/plots/accuracy_curve.png
  artifacts/plots/confusion_matrix.png
  artifacts/reports/leaf_scope_check.json
  artifacts/reports/comparison_row.csv
  artifacts/reports/classification_report.txt
  artifacts/reports/training_stage2.csv
  artifacts/reports/model_metrics.json
  artifacts/reports/training_history.json
  artifacts/reports/leakage_check.json
  artifacts/reports/confusion_matrix_test.csv
  artifacts/reports/training_stage1.csv
  artifacts/reports/test_probabilities.npy
  models/densenet121/leaf_reference.npz
  models/densenet121/best_densenet121.keras
  models/densenet121/class_names.json
  models/densenet121/class_names.npy


## After training — compare the three models

1. Run all three notebooks: **DenseNet121** (this one), **EfficientNetB0** and **ResNet50**.
2. Check that the **split fingerprint** printed in Step 5 is the **same** in all three notebooks. If it
   is not, the models were tested on different images and the comparison is not valid.
3. Check that Step 5b printed **NO DATA LEAKAGE FOUND** in all three.
4. Download the three `smart_crop_care_<model>_results.zip` files and run `compare_three_models.py` on
   your PC. It checks again that the three splits are identical and builds one comparison table, plus a
   paired test on the same test images.
5. Report the **test** numbers (Step 11). Don't report train or validation numbers as test results.

Using a model in the app needs its own preprocessing and `leaf_reference.npz`; ask before replacing any
project files.